# Tabular foundation model blend


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

raw_data_dir = project_root / "data" / "raw"
tfm_dir = project_root / "data" / "tfm"
oof_dir = project_root / "artifacts" / "oof"
test_predictions_dir = project_root / "artifacts" / "test_predictions"
metrics_dir = project_root / "artifacts" / "metrics"
submissions_dir = project_root / "submissions"

train_df = pd.read_csv(raw_data_dir / "train.csv", usecols=["satisfaction"])
sample_submission = pd.read_csv(raw_data_dir / "sample_submission.csv")
target = train_df["satisfaction"].astype("int8")

print("train rows:", len(target))
print("test rows:", len(sample_submission))
print("TFM files:", len(list(tfm_dir.glob("*.npy"))))

## Prediction validation


In [ ]:
model_scores = {}

for oof_path in sorted(tfm_dir.glob("oof_*.npy")):
    model_name = oof_path.stem.removeprefix("oof_")
    test_path = tfm_dir / f"test_{model_name}.npy"

    oof_prediction = np.load(oof_path).reshape(-1)
    test_prediction = np.load(test_path).reshape(-1)

    assert len(oof_prediction) == len(target)
    assert len(test_prediction) == len(sample_submission)
    assert np.isfinite(oof_prediction).all()
    assert np.isfinite(test_prediction).all()

    model_scores[model_name] = roc_auc_score(target, oof_prediction)

for name, score in sorted(
    model_scores.items(),
    key=lambda item: item[1],
    reverse=True,
):
    print(f"{name:12s} {score:.9f}")

## Rank ensemble


In [ ]:
def to_rank(values):
    return pd.Series(values).rank(pct=True).to_numpy(dtype="float32")

tabpfn_oof = np.load(tfm_dir / "oof_tabpfn35.npy").reshape(-1)
tabpfn_test = np.load(tfm_dir / "test_tabpfn35.npy").reshape(-1)
route_oof = np.load(oof_dir / "lightgbm_route_te.npy")
route_test = np.load(test_predictions_dir / "lightgbm_route_te.npy")
proper_oof = np.load(oof_dir / "proper_te.npy")
proper_test = np.load(test_predictions_dir / "proper_te.npy")

blend_oof = (
    0.70 * to_rank(tabpfn_oof)
    + 0.18 * to_rank(route_oof)
    + 0.12 * to_rank(proper_oof)
)

blend_test = (
    0.70 * to_rank(tabpfn_test)
    + 0.18 * to_rank(route_test)
    + 0.12 * to_rank(proper_test)
)

blend_auc = roc_auc_score(target, blend_oof)

np.save(oof_dir / "blend_tabpfn70_route18_proper12.npy", blend_oof)
np.save(
    test_predictions_dir / "blend_tabpfn70_route18_proper12.npy",
    blend_test,
)

metrics = {
    "source": "https://www.kaggle.com/datasets/cdeotte/s6e10-tfm-oof-and-test-predictions",
    "tabpfn35_oof_auc": float(model_scores["tabpfn35"]),
    "blend_oof_auc": float(blend_auc),
    "blend_rank_weights": {
        "tabpfn35": 0.70,
        "lightgbm_route_te": 0.18,
        "proper_te": 0.12,
    },
}

with open(metrics_dir / "tfm_blend.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4, ensure_ascii=False)

submission = sample_submission.copy()
submission["satisfaction"] = blend_test
submission_path = submissions_dir / "blend_tabpfn70_route18_proper12.csv"
submission.to_csv(submission_path, index=False)

print("TabPFN OOF AUC:", model_scores["tabpfn35"])
print("blend OOF AUC:", blend_auc)
print("saved:", submission_path)
submission.head()